**개념 정리**

앙상블: 일련의 예측기
- 앙상블 학습: 앙상블로부터 예측을 수집
- 앙상블 방법: 앙상블 학습 알고리즘
- 랜덤 포레스트: 결정 트리의 앙상블

투표 기반 분류기
- 직접 투표(hard voting): 각 분류기의 예측을 모아서 가장 많이 선택된 클래스 예측(다수결)
    - 다수결 투표 분류기가 앙상블에 포함된 개별 분류기 중 가장 뛰어난 것보다 정확도가 높을 수 있음
    - 각 분류기가 약한 학습기일지라도 충분하게 많고 다양하다면 앙상블은 강한 학습기가 될 수 있음
- 간접 투표(soft voting): 개별 분류기의 예측을 평균 내어 확률이 가장 높은 클래스를 예측
    - 확률이 높은 투표에 비중을 더 둠, 직접 투표 방식보다 성능이 높음

배깅과 페이스팅
- 배깅: 훈련 세트에서 중복을 허용하여 샘플링하는 방식
- 페이스팅: 중복을 허용하지 않고 샘플링하는 방식
- 수집 함수: 분류일 때는 통계적 최빈값, 회귀일 때는 평균
    - 개별 예측기는 수집 함수를 통과하면 평향과 분산이 모두 감소

사이킷런의 배깅과 페이스팅
- BaggingClassifier 제공
    - 기반이 되는 분류기가 결정 트리 분류기처럼 클래스 확률을 추정할 수 있으면 직접 투표 대신 자동으로 간접 투표 방식 사용
- 부트스트래핑은 각 예측기가 학습하는 서브셋에 다양성을 증가시키므로 배깅이 페이스팅보다 편향이 조금 더 높음

oob 평가
- 예측기가 훈련되는 동안 oob 샘플 사용 X, 별도의 검증 세트를 사용X, oob 샘플을 사용해 평가
- 앙상블의 평가는 각 예측기의 oob 평가를 평균하여 얻음
- BaggingClassifier를 만들 때, oob_score=True로 지정하면 훈련이 끝난 후 자동으로 수행
- oob_decision_function_: oob 샘플에 대한 결정 함수의 값 확인

랜덤 패치와 랜덤 서브스페이스
- BaggingClassifier의 샘플링은 max_features, bootstrap_features 두 매개변수로 조절
    - 각 예측기는 무작위로 선택한 입력 특성의 일부분으로 훈련
- 랜덤 패치 방식: 훈련 특성과 샘플을 모두 샘플링하는 것
- 랜덤 서브스페이스 방식: 훈련 샘플을 모두 사용하고 특성은 샘플링하는 것

랜덤 포레스트
- 전형적으로 max_samples를 훈련 세트의 크기로 지정
- RandomForestClassifier 사용
- 트리의 노드를 분할할 때 전체 특성 중 최선의 특성을 찾는 대신 무작위로 선택한 특성 후보 중에서 최적의 특성을 찾는 식으로 무작위성을 더 주입
    - 트리를 다양하게 만들고 편향을 손해보는 대신 분산을 낮춤

엑스트라 트리
- 익스트림 랜덤 트리: 극단적으로 무작위한 트리의 랜덤 포레스트 (=엑스트라 트리)
- 일반적인 랜덤 포레스트보다 훨씬 빠름
- ExtraTreesClassifier를 사용

특성 중요도
- 어떤 특성을 사용한 노드가 평균적으로 불순도를 얼마나 감소시키는지 확인하여 측정

**코드 필사**

In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
#import package
import numpy as np
import os

In [3]:
#5장에서의 moons dataset 불러오기
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
X, y=make_moons(n_samples=100, noise=0.15)
X_train, X_test, y_train, y_test=train_test_split(X, y, test_size=0.2)

투표 기반 분류기

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

log_clf=LogisticRegression()
rnd_clf=RandomForestClassifier()
svm_clf=SVC()

voting_clf=VotingClassifier(
    estimators=[('lr', log_clf), ('rf', rnd_clf), ('svc', svm_clf)],
    voting='hard')
voting_clf.fit(X_train, y_train)

VotingClassifier(estimators=[('lr', LogisticRegression()),
                             ('rf', RandomForestClassifier()), ('svc', SVC())])

In [5]:
from sklearn.metrics import accuracy_score
for clf in (log_clf, rnd_clf, svm_clf, voting_clf):
    clf.fit(X_train, y_train)
    y_pred=clf.predict(X_test)
    print(clf.__class__.__name__, accuracy_score(y_test, y_pred))

LogisticRegression 0.9
RandomForestClassifier 0.95
SVC 0.95
VotingClassifier 0.95


배깅과 페이스팅

In [6]:
from sklearn.ensemble import BaggingClassifier
from sklearn.tree import DecisionTreeClassifier

bag_clf=BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500,
    max_samples=50, bootstrap=True, n_jobs=-1)
bag_clf.fit(X_train, y_train)
y_pred=bag_clf.predict(X_test)

oob 평가

In [7]:
bag_clf=BaggingClassifier(
    DecisionTreeClassifier(), n_estimators=500,
    bootstrap=True, n_jobs=-1, oob_score=True)

bag_clf.fit(X_train, y_train)
bag_clf.oob_score_

0.925

In [8]:
from sklearn.metrics import accuracy_score
y_pred=bag_clf.predict(X_test)
accuracy_score(y_test, y_pred)

0.95

In [9]:
bag_clf.oob_decision_function_

array([[1.        , 0.        ],
       [0.        , 1.        ],
       [0.98058252, 0.01941748],
       [0.88023952, 0.11976048],
       [0.82417582, 0.17582418],
       [0.00558659, 0.99441341],
       [1.        , 0.        ],
       [1.        , 0.        ],
       [0.47120419, 0.52879581],
       [0.        , 1.        ],
       [0.80110497, 0.19889503],
       [0.        , 1.        ],
       [0.90860215, 0.09139785],
       [0.        , 1.        ],
       [0.        , 1.        ],
       [0.01169591, 0.98830409],
       [0.3559322 , 0.6440678 ],
       [0.00591716, 0.99408284],
       [0.70984456, 0.29015544],
       [0.        , 1.        ],
       [0.88505747, 0.11494253],
       [0.01052632, 0.98947368],
       [0.01807229, 0.98192771],
       [0.01036269, 0.98963731],
       [1.        , 0.        ],
       [0.02285714, 0.97714286],
       [0.        , 1.        ],
       [0.84023669, 0.15976331],
       [1.        , 0.        ],
       [0.93717277, 0.06282723],
       [0.

랜덤 포레스트

In [10]:
from sklearn.ensemble import RandomForestClassifier

rnd_clf=RandomForestClassifier(n_estimators=500, max_leaf_nodes=16, n_jobs=-1)
rnd_clf.fit(X_train, y_train)

y_pred_rf=rnd_clf.predict(X_test)

In [11]:
bag_clf=BaggingClassifier(
    DecisionTreeClassifier(max_features="auto", max_leaf_nodes=16),
    n_estimators=500, max_samples=1.0, bootstrap=True, n_jobs=-1)

특성 중요도

In [12]:
from sklearn.datasets import load_iris
iris=load_iris()
rnd_clf=RandomForestClassifier(n_estimators=500, n_jobs=-1)
rnd_clf.fit(iris["data"], iris["target"])
for name, score in zip(iris["feature_names"], rnd_clf.feature_importances_):
    print(name, score)

sepal length (cm) 0.09260953557350379
sepal width (cm) 0.023383703712850156
petal length (cm) 0.4325967890643697
petal width (cm) 0.4514099716492763
